# Agronia — Training Model Padi (Google Colab)

Notebook ini melatih classifier **4 kelas** untuk MVP Agronia:

1. Bacterial Leaf Blight
2. Rice Blast
3. Tungro
4. Healthy Leaf

**Dataset:** Rice Leaf and Crop Disease Detection Dataset (Labib dkk., 2024)  
**DOI:** https://doi.org/10.17632/g7tcwvshff.1  
**Lisensi:** CC BY 4.0

## Sebelum menjalankan

1. Unduh ZIP dataset dari halaman Mendeley (tombol **Download All**).
2. Unggah ZIP ke Google Drive di `MyDrive/agronia/rice_leaf.zip`.
3. Di Colab: **Runtime → Change runtime type → T4 GPU**.
4. Jalankan sel dari atas ke bawah.

ZIP dari Mendeley berisi ZIP bersarang; notebook menanganinya otomatis. Hanya gambar **original/raw** yang dipakai. Kelas `Rice` (generik) dan folder
`augmented` diabaikan agar evaluasi tidak bias. Dataset tidak di-commit ke repo.


## 1. Cek GPU dan TensorFlow


In [ ]:
!nvidia-smi
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))


## 2. Hubungkan Google Drive dan tentukan lokasi ZIP


In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

ZIP_PATH = Path("/content/drive/MyDrive/agronia/rice_leaf.zip")
WORK_DIR = Path("/content/agronia")
RAW_DIR = WORK_DIR / "extracted"
ART_DIR = WORK_DIR / "artifacts"
ART_DIR.mkdir(parents=True, exist_ok=True)

if not ZIP_PATH.is_file():
    raise FileNotFoundError(
        f"ZIP tidak ditemukan di {ZIP_PATH}. "
        "Unggah dataset ke Google Drive dengan path tersebut, "
        "atau ubah ZIP_PATH di sel ini."
    )

print("ZIP:", ZIP_PATH)
print(f"Ukuran: {ZIP_PATH.stat().st_size / 1e6:.1f} MB")


## 3. Ekstrak dan inspeksi struktur dataset

ZIP dari Mendeley berisi **ZIP di dalam ZIP**. Sel ini mengekstrak ZIP luar,
lalu otomatis mengekstrak ZIP dalam jika belum ada gambar. Struktur aslinya:

```
<Kelas>/orginal/<subfolder>/gambar.jpg
<Kelas>/augmented/...        <- diabaikan
```

Perhatikan typo `orginal` dari pembuat dataset; notebook menanganinya.


In [ ]:
import shutil
import zipfile
from collections import Counter

EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def extract_zip(path, destination):
    with zipfile.ZipFile(path) as archive:
        archive.extractall(destination)


def find_images(root):
    return [
        p
        for p in root.rglob("*")
        if p.is_file()
        and p.suffix.lower() in EXTS
        and "__MACOSX" not in p.parts
    ]


if RAW_DIR.exists():
    shutil.rmtree(RAW_DIR)
RAW_DIR.mkdir(parents=True)
extract_zip(ZIP_PATH, RAW_DIR)

DATA_ROOT = RAW_DIR
files = find_images(DATA_ROOT)

if not files:
    nested_zips = [p for p in RAW_DIR.rglob("*.zip") if p.is_file()]
    print("Tidak ada gambar langsung; ZIP bersarang ditemukan:", len(nested_zips))
    if nested_zips:
        DATA_ROOT = RAW_DIR / "dataset"
        DATA_ROOT.mkdir(parents=True, exist_ok=True)
        for nested_zip in nested_zips:
            print("Mengekstrak:", nested_zip.name)
            extract_zip(nested_zip, DATA_ROOT)
        files = find_images(DATA_ROOT)

if not files:
    raise SystemExit(
        "Tidak ada gambar setelah ekstraksi. Periksa isi ZIP dan laporkan struktur foldernya."
    )

print("Total gambar:", len(files))
counts = Counter(str(p.parent.relative_to(DATA_ROOT)) for p in files)
for folder, count in counts.most_common(30):
    print(f"{count:5d}  {folder}")


## 4. Petakan kelas asli dan abaikan augmentasi

Hanya folder `orginal`/`original`/`raw` yang dipakai. Kelas `Rice` (generik)
dan semua folder `augmented`/`aug` diabaikan agar evaluasi tidak bias.


In [ ]:
import hashlib

import pandas as pd
from PIL import Image

CLASS_ORDER = [
    "Bacterial Leaf Blight",
    "Healthy Leaf",
    "Rice Blast",
    "Tungro",
]
CLASS_IDS = {
    "Bacterial Leaf Blight": "padi-bacterial-blight",
    "Healthy Leaf": "padi-sehat",
    "Rice Blast": "padi-blast",
    "Tungro": "padi-tungro",
}
ALIASES = {
    "bacterial leaf blight": "Bacterial Leaf Blight",
    "bacterial blight": "Bacterial Leaf Blight",
    "bacterialleafblight": "Bacterial Leaf Blight",
    "bacterial leaf blight disease": "Bacterial Leaf Blight",
    "healthy leaf": "Healthy Leaf",
    "healthy leaves": "Healthy Leaf",
    "healthy": "Healthy Leaf",
    "rice blast": "Rice Blast",
    "blast": "Rice Blast",
    "leaf blast": "Rice Blast",
    "tungro": "Tungro",
}
ORIGINAL_NAMES = {"orginal", "original", "raw", "asli"}


def normalize(name):
    key = name.strip().lower().replace("_", " ").replace("-", " ")
    return " ".join(key.split())


def class_dirs(root):
    return sorted(
        d for d in root.iterdir() if d.is_dir() and d.name != "__MACOSX"
    )


top_dirs = class_dirs(DATA_ROOT)
if len(top_dirs) == 1:
    only = top_dirs[0]
    children = class_dirs(only)
    if (
        normalize(only.name) not in ALIASES
        and any(normalize(d.name) in ALIASES for d in children)
    ):
        print("Masuk ke folder pembungkus:", only.name)
        DATA_ROOT = only
        top_dirs = children

rows = []
ignored = Counter()
for class_dir in top_dirs:
    canonical = ALIASES.get(normalize(class_dir.name))
    raw_dirs = [
        d
        for d in class_dir.rglob("*")
        if d.is_dir() and normalize(d.name) in ORIGINAL_NAMES
    ]
    raw_images = [
        p
        for directory in raw_dirs
        for p in directory.rglob("*")
        if p.is_file() and p.suffix.lower() in EXTS and "__MACOSX" not in p.parts
    ]
    if canonical is None:
        ignored[class_dir.name] += len(raw_images)
        continue
    if not raw_images:
        print(f"PERINGATAN: {class_dir.name} tidak punya folder original/raw.")
        continue
    for image_path in raw_images:
        rows.append(
            {
                "path": str(image_path),
                "folder": class_dir.name,
                "label": canonical,
            }
        )

df = pd.DataFrame(rows)
print("\nGambar original dipakai:", len(df))
print("Gambar original diabaikan:", sum(ignored.values()), dict(ignored))
print("\nJumlah per kelas:")
print(df["label"].value_counts())

missing = [name for name in CLASS_ORDER if name not in set(df["label"])]
if missing:
    raise SystemExit(f"Kelas tidak ditemukan: {missing}. Periksa nama folder dataset.")
if df.empty:
    raise SystemExit("Tidak ada gambar yang bisa dipakai.")


def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


df["sha256"] = df["path"].map(sha256)

bad = []
for image_path in df["path"]:
    try:
        with Image.open(image_path) as image:
            image.verify()
    except Exception:
        bad.append(image_path)
if bad:
    print(f"\nFile rusak diabaikan: {len(bad)}")
    for image_path in bad[:10]:
        print(" -", image_path)
    df = df[~df["path"].isin(bad)]

duplicates = int(df["sha256"].duplicated().sum())
df = df.drop_duplicates("sha256", keep="first").reset_index(drop=True)
print(f"\nDuplikat persis dihapus: {duplicates}")
print("Gambar bersih:", len(df))
print(df["label"].value_counts())


## 5. Tampilkan contoh gambar per kelas


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(CLASS_ORDER), 4, figsize=(12, 12))
for row_index, label in enumerate(CLASS_ORDER):
    subset = df[df["label"] == label].head(4)
    for column, (_, item) in enumerate(subset.iterrows()):
        axis = axes[row_index][column]
        with Image.open(item["path"]) as image:
            axis.imshow(image.convert("RGB"))
        axis.set_title(label if column == 0 else "")
        axis.axis("off")
plt.tight_layout()
plt.show()


## 6. Split train/validation/test (stratified, tanpa kebocoran)


In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=42, stratify=df["label"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42, stratify=temp_df["label"]
)

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()
train_df["split"] = "train"
val_df["split"] = "validation"
test_df["split"] = "test"

split_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
split_df.to_csv(ART_DIR / "splits.csv", index=False)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print("\nPer kelas per split:")
print(pd.crosstab(split_df["label"], split_df["split"]))

train_hashes = set(train_df["sha256"])
val_hashes = set(val_df["sha256"])
test_hashes = set(test_df["sha256"])
assert not (train_hashes & val_hashes), "Kebocoran train/validation"
assert not (train_hashes & test_hashes), "Kebocoran train/test"
assert not (val_hashes & test_hashes), "Kebocoran validation/test"
print("\nTidak ada hash yang bocor antar split.")

for split_name, frame in [
    ("train", train_df),
    ("validation", val_df),
    ("test", test_df),
]:
    counts = frame["label"].value_counts()
    if counts.min() < 10:
        print(f"PERINGATAN: {split_name} punya kelas < 10 gambar.")


## 7. Pipeline tf.data dan augmentasi


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

IMG_SIZE = 224
BATCH_SIZE = 32
SEED = 42

label_to_index = {name: index for index, name in enumerate(CLASS_ORDER)}
index_to_label = {index: name for name, index in label_to_index.items()}


def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image.set_shape([IMG_SIZE, IMG_SIZE, 3])
    return image, label


def make_dataset(frame, training):
    paths = frame["path"].values
    labels = frame["label"].map(label_to_index).values.astype("int32")
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        dataset = dataset.shuffle(
            len(frame), seed=SEED, reshuffle_each_iteration=True
        )
    dataset = dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df, training=False)
test_ds = make_dataset(test_df, training=False)

data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.08),
        layers.RandomZoom(0.10),
        layers.RandomContrast(0.10),
    ],
    name="augmentation",
)

print("Batch train:", len(train_ds))
print("Batch validation:", len(val_ds))
print("Batch test:", len(test_ds))


## 8. Latih classifier (MobileNetV2, base dibekukan)


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

encoded_train = train_df["label"].map(label_to_index).values
weights = compute_class_weight(
    "balanced", classes=np.arange(len(CLASS_ORDER)), y=encoded_train
)
class_weight = {index: float(weight) for index, weight in enumerate(weights)}
print("Class weight:", class_weight)

base_model = keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="image")
x = data_augmentation(inputs)
x = keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.30)(x)
outputs = layers.Dense(len(CLASS_ORDER), activation="softmax", name="prediction")(x)

model = keras.Model(inputs, outputs, name="agronia_rice_mobilenetv2")
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

callbacks_stage1 = [
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=4, restore_best_weights=True
    ),
    keras.callbacks.ModelCheckpoint(
        ART_DIR / "best_stage1.keras",
        monitor="val_accuracy",
        save_best_only=True,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6
    ),
]

history_stage1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=12,
    class_weight=class_weight,
    callbacks=callbacks_stage1,
)


## 9. Fine-tuning lapisan terakhir


In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks_stage2 = [
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=4, restore_best_weights=True
    ),
    keras.callbacks.ModelCheckpoint(
        ART_DIR / "best_model.keras",
        monitor="val_accuracy",
        save_best_only=True,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7
    ),
]

history_stage2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
    class_weight=class_weight,
    callbacks=callbacks_stage2,
)


## 10. Kurva training


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for axis, metric in zip(axes, ["loss", "accuracy"]):
    axis.plot(history_stage1.history[metric], label=f"train {metric} (stage 1)")
    axis.plot(
        history_stage1.history[f"val_{metric}"],
        label=f"val {metric} (stage 1)",
    )
    axis.plot(
        history_stage2.history[metric],
        label=f"train {metric} (stage 2)",
        linestyle="--",
    )
    axis.plot(
        history_stage2.history[f"val_{metric}"],
        label=f"val {metric} (stage 2)",
        linestyle="--",
    )
    axis.set_xlabel("epoch")
    axis.set_ylabel(metric)
    axis.legend()
    axis.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(ART_DIR / "training_curves.png", dpi=150)
plt.show()


## 11. Evaluasi di test set


In [ ]:
import json
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

test_loss, test_accuracy = model.evaluate(test_ds)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

y_true = []
y_prob = []
for images, labels in test_ds:
    probabilities = model.predict(images, verbose=0)
    y_true.extend(labels.numpy().tolist())
    y_prob.extend(probabilities.tolist())

y_true = np.array(y_true)
y_prob = np.array(y_prob)
y_pred = y_prob.argmax(axis=1)

report = classification_report(
    y_true,
    y_pred,
    target_names=CLASS_ORDER,
    output_dict=True,
    zero_division=0,
)
print(classification_report(y_true, y_pred, target_names=CLASS_ORDER, zero_division=0))

matrix = confusion_matrix(y_true, y_pred, labels=np.arange(len(CLASS_ORDER)))
print("Confusion matrix:")
print(matrix)

fig, axis = plt.subplots(figsize=(7, 6))
image = axis.imshow(matrix, cmap="Blues")
axis.set_xticks(range(len(CLASS_ORDER)))
axis.set_yticks(range(len(CLASS_ORDER)))
axis.set_xticklabels(CLASS_ORDER, rotation=45, ha="right")
axis.set_yticklabels(CLASS_ORDER)
for row in range(matrix.shape[0]):
    for column in range(matrix.shape[1]):
        axis.text(column, row, matrix[row, column], ha="center", va="center")
axis.set_xlabel("Prediksi")
axis.set_ylabel("Aktual")
axis.set_title("Confusion Matrix — Test Set")
fig.colorbar(image, ax=axis)
plt.tight_layout()
plt.savefig(ART_DIR / "confusion_matrix.png", dpi=150)
plt.show()

metrics = {
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "test_loss": float(test_loss),
    "n_test": int(len(y_true)),
    "class_order": CLASS_ORDER,
    "classification_report": report,
    "confusion_matrix": matrix.tolist(),
}
with open(ART_DIR / "metrics.json", "w", encoding="utf-8") as handle:
    json.dump(metrics, handle, indent=2, ensure_ascii=False)
print("Metrik disimpan ke", ART_DIR / "metrics.json")


## 12. Simpan model dan classes.json


In [ ]:
MODEL_VERSION = "rice-mobilenetv2-v1"
model_path = ART_DIR / "agronia_rice_mobilenetv2.keras"
model.save(model_path)

classes_payload = {
    "model_version": MODEL_VERSION,
    "description": (
        "Classifier padi 4 kelas untuk MVP Agronia. "
        "Hanya gambar original/raw; kelas Rice dan augmentasi tidak dipakai."
    ),
    "image_size": IMG_SIZE,
    "preprocessing": "RGB, resize 224x224, skala 0-255 (preprocess_input di dalam model)",
    "dataset": {
        "name": "Rice Leaf and Crop Disease Detection Dataset",
        "doi": "10.17632/g7tcwvshff.1",
        "license": "CC BY 4.0",
        "citation": "Labib, R., Mim, S., Mojumdar, M. U. (2024)",
    },
    "classes": [
        {
            "index": index,
            "class_id": CLASS_IDS[label],
            "name": label,
            "name_id": {
                "Bacterial Leaf Blight": "Hawar Daun Bakteri",
                "Healthy Leaf": "Daun Sehat",
                "Rice Blast": "Blas (Blast)",
                "Tungro": "Tungro",
            }[label],
        }
        for index, label in enumerate(CLASS_ORDER)
    ],
}
with open(ART_DIR / "classes.json", "w", encoding="utf-8") as handle:
    json.dump(classes_payload, handle, indent=2, ensure_ascii=False)

print("Model:", model_path, f"({model_path.stat().st_size / 1e6:.1f} MB)")
print("Classes:", ART_DIR / "classes.json")
print("Upload folder ke repo: model -> ai/weights/, classes.json -> ai/classes.json")


## 13. Unduh hasil training


In [ ]:
import zipfile
from google.colab import files

bundle_path = ART_DIR / "agronia_rice_mobilenetv2_bundle.zip"
artifacts = [
    model_path,
    ART_DIR / "classes.json",
    ART_DIR / "metrics.json",
    ART_DIR / "confusion_matrix.png",
    ART_DIR / "training_curves.png",
    ART_DIR / "splits.csv",
]

with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for artifact in artifacts:
        if artifact.exists():
            archive.write(artifact, artifact.name)

print("Bundle:", bundle_path)
files.download(str(bundle_path))


## Setelah training

1. Ekstrak bundle di laptop.
2. Taruh model di `ai/weights/`.
3. Taruh `classes.json` di `ai/classes.json`.
4. Simpan `metrics.json`, `confusion_matrix.png`, `training_curves.png`, dan
   `splits.csv` untuk dokumentasi evaluasi.
5. Lanjut ke implementasi inference di `ai/service/main.py` dan pengujian
   end-to-end.

Catatan: akurasi test set hanya berlaku untuk data ini. Jangan klaim akurasi
lapangan sebelum diuji dengan foto asli dari Timor-Leste.
